#### SBERT
- BERT 모델 : 문장 이해용 Encoder
    - 문장 쌍 비교
- SBERT 모델 : 문장 의미 임베딩 
    - 벡터의 비교용 

In [1]:
# 라이브러리 설치 
# !pip install sentence-transformers

In [2]:
import torch 
from sentence_transformers import SentenceTransformer, util

c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 모델을 로드 -> 두개의 문장을 비교(코사인 유사도)
# 다목적 한국 SBERT
model_name = 'jhgan/ko-sroberta-multitask'
# 문장 유사도 특화 
model_name2 = 'BM-K/KoSimCSE-roberta-multitask'

sbert = SentenceTransformer(model_name)
sbert2 = SentenceTransformer(model_name2)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [4]:
# 최대 토큰의 길이를 설정 
sbert.max_seq_length = 256
sbert2.max_seq_length = 256

In [5]:
# 두개의 문장을 비교 
doc1 = "이 카메라는 색감이 자연스럽고 배터리도 오래간다"
doc2 = "배터리 성능이 좋고 사진 품질이 뛰어나다"

In [6]:
# 두개의 문장을 임베딩 -> 코사인 유사도 계산 
# sbert인 경우 
with torch.inference_mode():
    emb1 = sbert.encode(doc1, convert_to_tensor=True, normalize_embeddings=True)
    emb2 = sbert.encode(doc2, convert_to_tensor=True, normalize_embeddings=True)
# 코사인 유사도 계산
cos_sim = util.cos_sim(emb1, emb2).item()
print("유사도 : ", round(cos_sim, 4))

유사도 :  0.6948


In [7]:
# 두개의 문장을 임베딩 -> 코사인 유사도 계산 
# sbert2인 경우 
with torch.inference_mode():
    emb1 = sbert2.encode(doc1, convert_to_tensor=True, normalize_embeddings=True)
    emb2 = sbert2.encode(doc2, convert_to_tensor=True, normalize_embeddings=True)
# 코사인 유사도 계산
cos_sim = util.cos_sim(emb1, emb2).item()
print("유사도 : ", round(cos_sim, 4))

유사도 :  0.734


In [8]:
emb1.shape

torch.Size([768])

In [9]:
sentences = [
    '삼성전자 주가가 올랐다', 
    "코스피가 상승 마감했다", 
    '비가 많이 와서 항공편이 취소됬다'
]

with torch.inference_mode():
    embs = sbert2.encode(sentences, convert_to_tensor=True, 
                         normalize_embeddings=True)

# 3개의 문장에서의 유사도를 확인 
sim_metrix = util.cos_sim(embs, embs)
print(sim_metrix)

tensor([[1.0000, 0.3585, 0.0140],
        [0.3585, 1.0000, 0.1416],
        [0.0140, 0.1416, 1.0000]])


In [10]:
new_sentence = "증시가 강세였다"
# 임베딩 
new_emb = sbert2.encode(new_sentence, convert_to_tensor=True, 
                        normalize_embeddings=True)
# 유사도가 높은 상위의 n개 확인 
top_n = 2
hits = torch.topk(
    util.cos_sim(new_emb, embs).squeeze(0), k = top_n
)
hits

torch.return_types.topk(
values=tensor([0.6161, 0.6014]),
indices=tensor([0, 1]))

In [11]:
for score, idx in zip( hits.values.tolist(), hits.indices.tolist() ):
    print(f"{sentences[idx]} | score : {round(score, 3)}")

삼성전자 주가가 올랐다 | score : 0.616
코스피가 상승 마감했다 | score : 0.601


### 연습 
- ratings_train.txt 파일을 로드 
- 결측치 제거 
- document 컬럼의 문자 정규화(특수문자 제거, 2칸 이상의 공백 제거, 좌우 공백 제거) 
- 중복 document 제거 , 글자의 수가 1개 이하인 행은 제거  
- DataFrame에서 sample(n = 10000, random_state=42)로 임의의 데이터를 추출하여 저장 (head() -> 상위 데이터 | tail() -> 하위 데이터 | sample() -> 무작위 데이터) 
- train, test 셋으로 8:2 로 데이터분할
- sbert 모델은 'BM-K/KoSimCSE-roberta-multitask'을 이용
- Dataset을 정의 (Trainer 이용하지 않고 Dataset과 DataLoader 사용)
    - 입력받은 document와 label를 document는 SBERT 모델을 이용하여 인코딩 
    - label 데이터를 tensor형태로 변환 
    - `__len__` 함수는 라벨의 길이를 되돌려준다
    - `__getitem__` 함수는 인코딩된 데이터[idx], label[idx]를 되돌려준다
- Dataset를 train, test를 이용해서 Dataset을 생성 
- DataLoarder를 이용하여 배치의 사이즈는 128 shuffle은 True 구성한다. 

In [12]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [13]:
data = pd.read_csv("../data/ratings_train.txt", sep= '\t')
data.head(2)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1


In [14]:
data.isna().sum()

id          0
document    5
label       0
dtype: int64

In [15]:
len(data)

150000

In [16]:
data.dropna(inplace=True)

In [17]:
# 텍스트 정규화 함수
def normalize_token_text(text:str) -> str:
    text= re.sub(r'[^가-힣a-zA-Z0-9\s\.]', " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['document'] = data['document'].map(normalize_token_text)

In [18]:
data.drop_duplicates(subset=['document'], inplace=True)
data = data[data['document'].map(lambda x: len(x) > 1)]

In [19]:
samples = data.sample(n=10000, random_state=42)

In [20]:
x = samples['document'].values
y = samples['label'].values

In [21]:
X_train, X_test, Y_train, Y_test = train_test_split(x, y ,test_size=0.2, random_state=42, stratify=samples['label'])

In [22]:
model_name3 = 'BM-K/KoSimCSE-roberta-multitask'

sbert3 = SentenceTransformer(model_name3) 

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [23]:
# 딥러닝에서 사용할 데이터를 파이토치에 맞게 변환
class TextDataset(Dataset):
    def __init__(self, xs, ys):
        # 독립변수를 객체 안에 저장
        self.xs = xs
        # 종속변수를 객체 안에 저장
        self.ys = ys
    def __len__(self):
        return len(self.xs)
    def __getitem__(self, idx):
        return self.xs[idx], self.ys[idx]

In [24]:
train_ds = DataLoader(
    TextDataset(X_train, Y_train),
    batch_size=128,
    shuffle=True
)

test_ds = DataLoader(
    TextDataset(X_test, Y_test),
    batch_size=128,
    shuffle=True,
)

In [25]:
train_ds

In [26]:
test_ds

-------------

## 강사님 Ver

In [27]:
import re
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer

In [28]:
# 데이터 로드
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [29]:
# 결측치를 제거
df.dropna(subset='document', inplace=True)

In [30]:
# 텍스트 정규화 함수
def noramlize(text):
    text= re.sub(r'[^가-힣a-zA-Z0-9\s\.]', " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['document'] = df['document'].map(noramlize)

In [31]:
# 중복 데이터를 제거
df.drop_duplicates(subset='document', inplace=True)

In [32]:
# 문자열의 길이가 1 이하는 제거 -> 1초과인 데이터만 확인
flag = df['document'].str.len() > 1
df.loc[flag, ]

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요 한국인은 거들먹거리고 필리핀 혼혈은 착하다,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


In [33]:
# 랜덤한 데이터 10000개를 추출
df = df.sample(n=10000, random_state=42).reset_index(drop=True)

In [34]:
# train, test 분할
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['label']
)

In [35]:
model_name3 = 'BM-K/KoSimCSE-roberta-multitask'

sbert3 = SentenceTransformer(model_name3) 

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [41]:
# Dataset 정의 
class SBERTDataset(Dataset):
    # 생성자 함수 -> document, labels 받아와서 document 임베딩, labels는 tensor화
    def __init__(self, document, labels):
        # no_grad() -> 자동 미분 일시 정지 
        # inference_mode() -> 추론 모드 
        with torch.inference_mode():
            # 로드한 모델을 이용해서 encode 작업 
            # convet_to_tensor -> 결과값을 tensor로 받을것인가? (False : list)
            # normalize_embeddings -> L2 정규화 할것인가?
            self.emb = sbert3.encode(
                document, convert_to_tensor = True, normalize_embeddings=True
            )
        # labels를 tensor화 
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        # labels의 길이를 되돌려준다.
        return len(self.labels)
    def __getitem__(self, idx):
        return self.emb[idx], self.labels[idx]

In [43]:
# Dataset의 형태로 데이터프레임을 변환
train_ds = SBERTDataset(train_df['document'].tolist(), train_df['label'].tolist())
test_ds = SBERTDataset(test_df['document'].tolist(), test_df['label'].tolist())

In [44]:
# DataLoader를 이용해서 배치 사이즈만큼의 데이터를 생성
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=128, shuffle=True)

In [47]:
# 분류 모델 정의 (입력받눈 데이터 -> document tensor, label tensor)
class MLPHead(nn.Module):
    # 선형 모델에 데이터를 입력하는 형태
    # 선형 모델을 정의할때 인자값 (Linear(입력데이터의 피쳐의 수, 출력의 피쳐의 수))
    def __init__(self, input_dim, hidden=256, num_classes=2):
        super().__init__()
        # 다중 퍼셉트론층 구성
        self.net = nn.Sequential(
            # 선형 모델
            nn.Linear(input_dim, hidden),   # 입력 768 차원에서 출력은 256 차원
            nn.ReLU(),                      # 비선형 구조를 이해하기 위한 작업
            nn.Dropout(0, 2),               # 과적합 방지를 위한 소실 작업
            
        )
    # 순전파 함수 -> 독립변수를 받아서 예측 값을 되돌려준다.
    def forward(self, x):
        # x : 독립 변수 (document 데이터를 임베딩하고 배치로 묶은 데이터)
        result = self.net(x)    # 출력이 2차원인 확률 데이터
        return result


In [50]:
# MLPHead class를 생성하기 위해서는 input_dim 매개변수에 인자는 필수 항목
# input_dim -> 입력 데이터(독립변수)의 피쳐의 수를 의미
# 입력데이터 -> sbert3모델에서 임베딩이 된 독립변수의 피쳐의 수
# sbert3에서 설정이 된 출력 피쳐의 수를 변수에 저장
in_dim = sbert3.get_sentence_embedding_dimension()  # 출력 피쳐의 수를 되돌려주는 내장함수
in_dim

768

In [49]:
# MLPHead 모델 생성
clf = MLPHead(in_dim)
# 손실함수 -> 예측값과 실제값의 차이를 확인하는 함수
crit = nn.CrossEntropyLoss()
# 옵티마이저
opt = torch.optim.AdamW(clf.parameters(), lr= 2e-4)

In [53]:
# 학습 루프
# 학습 모드 전환
clf.train()

for epoch in range(5):
    total = 0.0
    for x, y in train_dl:
        # x : document데이터가 임베딩 벡터가 된 묶음 (tensor)
        # y : labels데이터가 tensor혀앹 묶음
        opt.zero_grad()
        # 순전파
        logits = clf(x)
        # 손실 계산 (예측값, 실제값)
        loss = crit(logits, y)
        # 역전파
        loss.backward()
        
        # 스탭
        opt.step()
        total += loss.item() * float(x.size(0))
    print(f"epoch : {epoch}, loss : {total/len(train_ds)}")

epoch : 0, loss : 5.476724502563477
epoch : 1, loss : 5.292230247497558
epoch : 2, loss : 5.114732063293457
epoch : 3, loss : 4.942318355560302
epoch : 4, loss : 4.7724157905578615


In [55]:
# 테스트 데이터를 이용하여 정확도, f1_score 확인
clf.eval()


y_true, y_pred = [], []


with torch.inference_mode():
    for x, y in test_dl:
        logits = clf(x)     # 예측 데이터 -> [0.xxx, 0.xxx]
        pred = logits.argmax(dim=1).tolist()     # 예측 데이터 -> [0, 1, 1, 0, ...]
        # y_true에 y를 리스트의 형태로 변환하고 데이터를 학장시킨다
        y_true.extend(y.tolist())
        y_pred += pred
print(y_true)
print(y_pred)

[0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 

In [56]:
print("accuracy_score", accuracy_score(y_true, y_pred))
print("f1_score", f1_score(y_true, y_pred))

accuracy_score 0.7565
f1_score 0.7508951406649617
